# The Task description
A supervised learning, binary classification task on an image dataset.

# Important Necessary Libs

In [1]:
import sys
import platform
import os

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Machine:", platform.machine())

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:22:19) [Clang 14.0.6 ]
Executable: /opt/anaconda3/bin/python
Platform: macOS-15.7.4-arm64-arm-64bit
Machine: arm64


In [5]:
%pip install --upgrade pip setuptools wheel

%pip install \
numpy==1.26.4 \
scipy==1.11.4 \
matplotlib==3.8.4 \
seaborn==0.13.2 \
pandas==2.2.2 \
scikit-learn==1.5.1 \
opencv-python==4.10.0.84 \
tensorflow==2.16.1 \
keras==3.3.3 \
kagglehub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kagglehub]
Note: you may need to restart the kernel to use updated packages.


In [13]:
import numpy as np
import scipy
import matplotlib
import seaborn as sns
import pandas as pd
import sklearn
import cv2
import kagglehub
import shutil

"""
requirements = {
    "numpy": ("1.24.0", np.__version__),
    "scipy": ("1.14.1", scipy.__version__),
    "matplotlib": ("3.9.2", matplotlib.__version__),
    "seaborn": ("0.12.0", sns.__version__),
    "pandas": ("2.2.3", pd.__version__),
    "scikit-learn": ("1.5.2", sklearn.__version__),
    "opencv-python": ("4.5.4", cv2.__version__),  # use prefix, not full wheel tag
}

for pkg, (expected, actual) in requirements.items():
    assert actual.startswith(expected), f"{pkg}: expected {expected}, got {actual}"

print("All checked versions match.")
"""

'\nrequirements = {\n    "numpy": ("1.24.0", np.__version__),\n    "scipy": ("1.14.1", scipy.__version__),\n    "matplotlib": ("3.9.2", matplotlib.__version__),\n    "seaborn": ("0.12.0", sns.__version__),\n    "pandas": ("2.2.3", pd.__version__),\n    "scikit-learn": ("1.5.2", sklearn.__version__),\n    "opencv-python": ("4.5.4", cv2.__version__),  # use prefix, not full wheel tag\n}\n\nfor pkg, (expected, actual) in requirements.items():\n    assert actual.startswith(expected), f"{pkg}: expected {expected}, got {actual}"\n\nprint("All checked versions match.")\n'

# Data Preprocessing

## Download Dataset

In [15]:
def ensure_pneumonia_dataset(dataset_name="paultimothymooney/chest-xray-pneumonia",
                             target_dir="data/chest_xray"):

    # get paths 
    project_root = os.getcwd()
    target_path = os.path.join(project_root, target_dir)

    # check if dataset already exists
    if os.path.exists(target_path):
        print(f"Dataset already exists at: {target_path}")
        return target_path

    print("Dataset not found. Downloading...")

    # 2️⃣ Download dataset
    path = kagglehub.dataset_download(dataset_name)

    # create parent folder if needed
    os.makedirs(os.path.dirname(target_path), exist_ok=True)

    # Move dataset into project
    shutil.move(path, target_path)

    print(f"Dataset stored at: {target_path}")
    return target_path

In [ ]:
ensure_pneumonia_dataset()

Dataset not found. Downloading...


100%|██████████████████████████████████████| 2.29G/2.29G [01:14<00:00, 33.2MB/s]

Extracting files...


## Read Data

In [11]:
# # Load all data, takes a long time hence single cell.
# for dirpath, dirnames, filenames in os.walk(path):
#     for filename in filenames:
#         print(os.path.join(dirpath, filename))

In [12]:
# There is a directory_name, three folder names, images types and format
BASE_DIR = "/kaggle/input/chest-xray-pneumonia/chest_xray/"
folder_names = ("train", "test", "val")
image_format = "jpeg"

# The directories to folder_types
TRAIN_DIR = os.path.join(BASE_DIR, folder_names[0])
TEST_DIR = os.path.join(BASE_DIR, folder_names[1])
VAL_DIR = os.path.join(BASE_DIR, folder_names[2])

print(

image = cv2.imread("/kaggle/input/chest-xray-pneumonia/chest_xray/train/NORMAL/NORMAL2-IM-0512-0001.jpeg")
print(image.shape)
cv2.imshow(image)

[ WARN:0@242.722] global loadsave.cpp:241 findDecoder imread_('/kaggle/input/chest-xray-pneumonia/chest_xray/train/NORMAL/NORMAL2-IM-0512-0001.jpeg'): can't open/read file: check file path/integrity


AttributeError: 'NoneType' object has no attribute 'shape'

In [91]:
IMG_SIZE = (1229, 1376) # WHy this image size?

# uint -> unsigned it and small as well since binary values
label_map = {
    "NORMAL": np.uint8(0),
    "PNEUMONIA": np.uint8(1)
}

def load_split(split_dir, img_size=IMG_SIZE):
    X = []
    y = [] # These are the supervised learning labels given based on the file structure indication of the outcome.

    for class_name in ["NORMAL", "PNEUMONIA"]:
        class_dir = os.path.join(split_dir, class_name)
        label = label_map[class_name]

        for fname in os.listdir(class_dir):
            fpath = os.path.join(class_dir, fname)

            if not os.path.isfile(fpath):
                continue

            img = cv2.imread(fpath, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # image unification
            img = cv2.resize(img, img_size)
            img = img.astype(np.float32) / 255.0

            X.append(img)
            y.append(label)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.uint8)

    # add channel dimension for CNNs
    X = np.expand_dims(X, axis=-1)

    return X, y

In [ ]:
# Load data into X and y (label outcome)
X_train, y_train = load_split(TRAIN_DIR)
X_test, y_test = load_split(TEST_DIR)
X_val, y_val = load_split(VAL_DIR)

## Sanity Check Data

In [ ]:
print("-----------------Sanity Checks-----------------")
# Equal amount of rows and columns
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert X_val.shape[0] == y_val.shape[0]

# Not empty
assert len(X_train) is not None
assert len(X_test) is not None
assert len(X_val) is not None

assert len(y_train) is not None
assert len(y_test) is not None
assert len(y_val) is not None

# Correct data type
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_val.dtype == np.float32


assert y_train.dtype == np.uint8
assert y_test.dtype == np.uint8
assert y_val.dtype == np.uint8


print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("-----------------Sanity Checks PASSED----------------\n")

## Exploratory Data Analysis

In [ ]:
from google.colab.patches import cv2_imshow

print(X_train[0])

# Normal images view
print(cv2_imshow(X_train[1]))



# Pneumonia images view



# Distributions

  # In train data

  # In test data

  # In val data

## Missing Value Treaments

## Outlier Treatments

## Duplicates and Garbage Value Treatment

## Normalisation

## Encoding

# Baseline

# Hyperparameter Optimisation and Alternative Model.

# Results generation